In [15]:
import os
import json

import boto3

'''
{
  "Effect": "Allow",
  "Action": [
    "ssm:DescribeInstancePatchStates",
    "ssm:DescribeInstancePatches",
    "ssm:DescribeInstancePatchStatesForPatchGroup",
    "ssm:ListComplianceItems",
    "ssm:ListComplianceSummaries",
    "ssm:GetComplianceDetailsByResource"
  ],
  "Resource": "*"
}
'''

basedirectory = os.getcwd()
aws_cred_file = os.path.join(basedirectory, "CONFIG", "Credentials.json")

with open(aws_cred_file) as f:
    aws_cred = json.load(f)
    aws_access_key_id = aws_cred["AWS_account_cred"]["CL_DEV"]["aws_access_key_id"]
    aws_secret_access_key = aws_cred["AWS_account_cred"]["CL_DEV"]["aws_secret_access_key"]



def vm_connection(resourse:str, region:str, access_key:str, secret_key:str):
    print("AWS Connection Establihing...")
    return boto3.client(
    resourse,
    region_name=region,    
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key
)


ec2_client = vm_connection("ec2", "us-west-2", aws_access_key_id, aws_secret_access_key)
ssm_client = vm_connection("ssm", "us-west-2", aws_access_key_id, aws_secret_access_key)
print(ec2_client)
print(ssm_client)

AWS Connection Establihing...
AWS Connection Establihing...


## GET EC2 Instance details

In [27]:
# Verify the connection by calling an actual AWS API.
# describe_instances returns details about EC2 instances in the region.
response = ec2_client.describe_instances()

instances = []
# for reservation in response["Reservations"]:
#     for instance in reservation["Instances"]:
#         instances.append({
#             "InstanceId": instance["InstanceId"],
#             "State": instance["State"]["Name"],
#             "Type": instance["InstanceType"],
#             "PrivateIp": instance.get("PrivateIpAddress"),
#             "PublicIp": instance.get("PublicIpAddress"),
#         })

# print(f"Found {len(instances)} instance(s):")
# print(json.dumps(instances, indent=4))

print(f"Found {len(response['Reservations'])} reservation(s):")
print(json.dumps(response, indent=4, default=str))

with open(os.path.join(basedirectory, "PATCH-INFO", "instance_ids_from_ec2_client.json"), "w") as f:
    json.dump(response, f, indent=4, default=str)

Found 4 reservation(s):
{
    "Reservations": [
        {
            "ReservationId": "r-0c5e6bb4fc884867e",
            "OwnerId": "117484691099",
            "RequesterId": "658754138699",
            "Groups": [],
            "Instances": [
                {
                    "Architecture": "x86_64",
                    "BlockDeviceMappings": [
                        {
                            "DeviceName": "/dev/sda1",
                            "Ebs": {
                                "AttachTime": "2024-02-20 21:59:26+00:00",
                                "DeleteOnTermination": true,
                                "Status": "attached",
                                "VolumeId": "vol-0f6b24b9469fd1b94",
                                "EbsCardIndex": 0
                            }
                        }
                    ],
                    "ClientToken": "d494bd37-7637-5b34-8d8d-55937b73b6ea",
                    "EbsOptimized": true,
                    "En

## GET describe_instance_patch_states details

In [29]:
# Collect instance IDs from the EC2 describe_instances response.
instance_ids = [
    instance["InstanceId"]
    for reservation in response["Reservations"]
    for instance in reservation["Instances"]
]

# describe_instance_patch_states requires InstanceIds (max 50 per call).
patch_states = ssm_client.describe_instance_patch_states(InstanceIds=instance_ids)
print(f"Found {len(patch_states['InstancePatchStates'])} instance patch state(s):")
print(json.dumps(patch_states, indent=4, default=str))



with open(os.path.join(basedirectory, "PATCH-INFO", "describe_instance_patch_states.json"), "w") as f:
    json.dump(patch_states, f, indent=4, default=str)






Found 4 instance patch state(s):
{
    "InstancePatchStates": [
        {
            "InstanceId": "i-00d3065b84292207f",
            "PatchGroup": "",
            "BaselineId": "pb-07b02614ef953c5d1",
            "SnapshotId": "3e852f07-2b5d-4545-be1b-840c03d3db45",
            "InstalledCount": 454,
            "InstalledOtherCount": 3,
            "InstalledPendingRebootCount": 98,
            "InstalledRejectedCount": 0,
            "MissingCount": 0,
            "FailedCount": 1,
            "UnreportedNotApplicableCount": -1,
            "NotApplicableCount": 11001,
            "AvailableSecurityUpdateCount": -1,
            "OperationStartTime": "2026-06-20 14:35:39+05:30",
            "OperationEndTime": "2026-06-20 14:43:54+05:30",
            "Operation": "Install",
            "LastNoRebootInstallOperationTime": "2026-06-20 14:43:54+05:30",
            "RebootOption": "NoReboot",
            "CriticalNonCompliantCount": 99,
            "SecurityNonCompliantCount": 45,
     

## GET  describe_instance_patches Details

In [35]:
# ssm:DescribeInstancePatches = describe_instance_patches(InstanceId=...)
# Takes ONE InstanceId per call, so loop over each instance.
for instance_id in instance_ids:
    patches = ssm_client.describe_instance_patches(InstanceId=instance_id)
    print(f"Instance {instance_id}: {len(patches['Patches'])} patch(es)")
    print(json.dumps(patches, indent=4, default=str))
    with open(os.path.join(basedirectory, "PATCH-INFO", "describe_instance_patches",f"describe_instance_patches_{instance_id}.json"), "w") as f:
        json.dump(patches, f, indent=4, default=str)

Instance i-09c4a0d8684e42d98: 50 patch(es)
{
    "Patches": [
        {
            "Title": "NetworkManager.x86_64:1:1.40.16-20.0.1.el8_10",
            "KBId": "NetworkManager.x86_64",
            "Classification": "Bugfix",
            "Severity": "",
            "State": "Installed",
            "InstalledTime": "2025-10-15 23:39:20+05:30"
        },
        {
            "Title": "NetworkManager-libnm.x86_64:1:1.40.16-20.0.1.el8_10",
            "KBId": "NetworkManager-libnm.x86_64",
            "Classification": "Bugfix",
            "Severity": "",
            "State": "Installed",
            "InstalledTime": "2025-10-15 23:39:20+05:30"
        },
        {
            "Title": "NetworkManager-team.x86_64:1:1.40.16-20.0.1.el8_10",
            "KBId": "NetworkManager-team.x86_64",
            "Classification": "Bugfix",
            "Severity": "",
            "State": "Installed",
            "InstalledTime": "2025-10-15 23:39:21+05:30"
        },
        {
            "Title": 

## GET describe_instance_patch_states_for_patch_group Details


In [ ]:

# ssm:DescribeInstancePatchStatesForPatchGroup -> describe_instance_patch_states_for_patch_group(PatchGroup=...)
# Requires the name of a patch group (the value of the "Patch Group" tag on instances).
patch_group = "YOUR_PATCH_GROUP_NAME"
group_states = ssm_client.describe_instance_patch_states_for_patch_group(PatchGroup=patch_group)
print(f"Patch group '{patch_group}': {len(group_states['InstancePatchStates'])} instance state(s)")
print(json.dumps(group_states, indent=4, default=str))

with open(os.path.join(basedirectory, "PATCH-INFO", "describe_instance_patch_states_for_patch_group.json"), "w") as f:
    json.dump(group_states, f, indent=4, default=str)

Patch group 'YOUR_PATCH_GROUP_NAME': 0 instance state(s)
{
    "InstancePatchStates": [],
    "ResponseMetadata": {
        "RequestId": "22f839ff-52db-4eae-8dd5-240438446a7f",
        "HTTPStatusCode": 200,
        "HTTPHeaders": {
            "server": "Server",
            "date": "Fri, 26 Jun 2026 10:27:44 GMT",
            "content-type": "application/x-amz-json-1.1",
            "content-length": "26",
            "connection": "keep-alive",
            "x-amzn-requestid": "22f839ff-52db-4eae-8dd5-240438446a7f",
            "cache-control": "no-store"
        },
        "RetryAttempts": 0
    }
}


## GET list_compliance_items Details

In [30]:

# ssm:ListComplianceItems  = list_compliance_items(ResourceIds=..., ResourceTypes=...)
compliance_items = ssm_client.list_compliance_items(
    ResourceIds=instance_ids,
    ResourceTypes=["ManagedInstance"],
)
print(f"Found {len(compliance_items['ComplianceItems'])} compliance item(s):")
print(json.dumps(compliance_items, indent=4, default=str))

with open(os.path.join(basedirectory, "PATCH-INFO", "list_compliance_items.json"), "w") as f:
    json.dump(compliance_items, f, indent=4, default=str)

Found 50 compliance item(s):
{
    "ComplianceItems": [
        {
            "ComplianceType": "Association",
            "ResourceType": "ManagedInstance",
            "ResourceId": "i-00d3065b84292207f",
            "Id": "11692cc7-aefd-4db8-b96f-016cb62cb20d",
            "Title": "",
            "Status": "COMPLIANT",
            "Severity": "UNSPECIFIED",
            "ExecutionSummary": {
                "ExecutionTime": "2026-06-26 06:40:09+05:30"
            },
            "Details": {
                "DocumentName": "AWS-GatherSoftwareInventory",
                "DocumentVersion": "1"
            }
        },
        {
            "ComplianceType": "Association",
            "ResourceType": "ManagedInstance",
            "ResourceId": "i-00d3065b84292207f",
            "Id": "a32ab54d-f5e7-4597-8cd1-f860327a2df2",
            "Title": "",
            "Status": "COMPLIANT",
            "Severity": "UNSPECIFIED",
            "ExecutionSummary": {
                "ExecutionTime":

## GET list_compliance_summaries Details

In [31]:

# ssm:ListComplianceSummaries = list_compliance_summaries()
compliance_summaries = ssm_client.list_compliance_summaries()
print(f"Found {len(compliance_summaries['ComplianceSummaryItems'])} compliance summary item(s):")
print(json.dumps(compliance_summaries, indent=4, default=str))

with open(os.path.join(basedirectory, "PATCH-INFO", "list_compliance_summaries.json"), "w") as f:
    json.dump(compliance_summaries, f, indent=4, default=str)


Found 3 compliance summary item(s):
{
    "ComplianceSummaryItems": [
        {
            "ComplianceType": "FleetTotal",
            "CompliantSummary": {
                "CompliantCount": 0,
                "SeveritySummary": {
                    "CriticalCount": 0,
                    "HighCount": 0,
                    "MediumCount": 0,
                    "LowCount": 0,
                    "InformationalCount": 0,
                    "UnspecifiedCount": 0
                }
            },
            "NonCompliantSummary": {
                "NonCompliantCount": 4,
                "SeveritySummary": {
                    "CriticalCount": 3,
                    "HighCount": 0,
                    "MediumCount": 0,
                    "LowCount": 0,
                    "InformationalCount": 0,
                    "UnspecifiedCount": 1
                }
            }
        },
        {
            "ComplianceType": "Association",
            "CompliantSummary": {
                "

In [34]:

# SSM refers to instances as "ManagedInstance".
for instance_id in instance_ids:
    response = ssm_client.list_compliance_items(
        ResourceIds=[instance_id],
        ResourceTypes=["ManagedInstance"],
        Filters=[
            {"Key": "ComplianceType", "Values": ["Patch"], "Type": "EQUAL"},
        ],
    )
    items = response["ComplianceItems"]
    print(f"Instance {instance_id}: {len(items)} patch compliance item(s)")
    print(json.dumps(response, indent=4, default=str))

    with open(os.path.join(basedirectory, "PATCH-INFO", "compliance-item-by-resource-instance-id",f"list_compliance_items_by_resource_{instance_id}.json"), "w") as f:
        json.dump(response, f, indent=4, default=str)


Instance i-09c4a0d8684e42d98: 50 patch compliance item(s)
{
    "ComplianceItems": [
        {
            "ComplianceType": "Patch",
            "ResourceType": "ManagedInstance",
            "ResourceId": "i-09c4a0d8684e42d98",
            "Id": "NetworkManager.x86_64",
            "Title": "NetworkManager.x86_64:1:1.40.16-20.0.1.el8_10",
            "Status": "COMPLIANT",
            "Severity": "CRITICAL",
            "ExecutionSummary": {
                "ExecutionTime": "2025-11-08 15:39:37+05:30",
                "ExecutionId": "73ef8079-99a3-4c5f-be95-4c123b6af415",
                "ExecutionType": "Command"
            },
            "Details": {
                "Classification": "Bugfix",
                "InstalledTime": "2025-10-15T18:09:20Z",
                "PatchBaselineId": "pb-07b02614ef953c5d1",
                "PatchState": "Installed"
            }
        },
        {
            "ComplianceType": "Patch",
            "ResourceType": "ManagedInstance",
            "